In [1]:
import pandas as pd
from urllib.parse import quote
import os

# Create folders
os.makedirs("../Data/raw", exist_ok=True)
os.makedirs("../Data/processed", exist_ok=True)


def fetch_all_rows(base_url, start_date, end_date, batch_size=50000):
    all_parts = []
    offset = 0

    where = (
        f"crash_date between '{start_date}T00:00:00' "
        f"and '{end_date}T23:59:59'"
    )
    encoded_where = quote(where)

    while True:
        url = (
            f"{base_url}"
            f"?$where={encoded_where}"
            f"&$limit={batch_size}"
            f"&$offset={offset}"
        )

        print(f"Downloading rows starting at offset {offset}...")
        chunk = pd.read_csv(url)

        if chunk.empty:
            break

        all_parts.append(chunk)

        if len(chunk) < batch_size:
            break

        offset += batch_size

    return pd.concat(all_parts, ignore_index=True)


# Fetch data
crashes = fetch_all_rows(
    "https://data.cityofchicago.org/resource/85ca-t3if.csv",
    "2024-01-01",
    "2024-12-31"
)

vehicles = fetch_all_rows(
    "https://data.cityofchicago.org/resource/68nd-jvt3.csv",
    "2024-01-01",
    "2024-12-31"
)

print("Crashes shape:", crashes.shape)
print("Vehicles shape:", vehicles.shape)


#raw data
crashes.to_csv("../Data/raw/crashes_2024_raw.csv", index=False)
vehicles.to_csv("../Data/raw/vehicles_2024_raw.csv", index=False)


# working copies for data cleaning
crashes_clean = crashes.copy()
vehicles_clean = vehicles.copy()


crashes_clean.to_csv("../Data/processed/crashes_2024_processed.csv", index=False)
vehicles_clean.to_csv("../Data/processed/vehicles_2024_processed.csv", index=False)

/var/folders/lx/_hl05hl904x2lrrmzp2l4b440000gn/T/ipykernel_52922/2767666066.py:29: DtypeWarning: Columns (31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  chunk = pd.read_csv(url)


/var/folders/lx/_hl05hl904x2lrrmzp2l4b440000gn/T/ipykernel_52922/2767666066.py:29: DtypeWarning: Columns (18,39,40,41,47,49,57,58,60,70) have mixed types. Specify dtype option on import or set low_memory=False.
  chunk = pd.read_csv(url)


/var/folders/lx/_hl05hl904x2lrrmzp2l4b440000gn/T/ipykernel_52922/2767666066.py:29: DtypeWarning: Columns (18,20,39,40,41,47,49,57,58,59,60,70) have mixed types. Specify dtype option on import or set low_memory=False.
  chunk = pd.read_csv(url)


/var/folders/lx/_hl05hl904x2lrrmzp2l4b440000gn/T/ipykernel_52922/2767666066.py:29: DtypeWarning: Columns (18,20,39,40,41,47,48,49,57,58,60,70) have mixed types. Specify dtype option on import or set low_memory=False.
  chunk = pd.read_csv(url)


/var/folders/lx/_hl05hl904x2lrrmzp2l4b440000gn/T/ipykernel_52922/2767666066.py:29: DtypeWarning: Columns (39,40,41,47,48,49,57,58,59,60) have mixed types. Specify dtype option on import or set low_memory=False.
  chunk = pd.read_csv(url)


/var/folders/lx/_hl05hl904x2lrrmzp2l4b440000gn/T/ipykernel_52922/2767666066.py:29: DtypeWarning: Columns (18,47,57,58,70) have mixed types. Specify dtype option on import or set low_memory=False.
  chunk = pd.read_csv(url)


Crashes shape: (112053, 48)
Vehicles shape: (228310, 71)


In [92]:
# First problem encountered, dataset is massive, need to be slective, selcted 2024, still massive
# solved by collecting by chunking to more easliy extract, only one request statement, download and work with it locally.

#Takes 2-3 minutes to aquire data for 2024, mention maybe in report, 
#we are going to use 2024 data because it is complete, 2025 can still be updated
#We are going to have file storage currently of Main folder, --> data folder, Notebook, scripts
#In data file there are currently 4 things, raw vehicledata, raw crash data, cleaning vehicle data, cleaning crash data.


print("Crashes shape:", crashes.shape)
print("Vehicles shape:", vehicles.shape)

Crashes shape: (112053, 48)
Vehicles shape: (228310, 71)


In [93]:
print(vehicles.columns.tolist())

['crash_unit_id', 'crash_record_id', 'crash_date', 'unit_no', 'unit_type', 'num_passengers', 'vehicle_id', 'cmrc_veh_i', 'make', 'model', 'lic_plate_state', 'vehicle_year', 'vehicle_defect', 'vehicle_type', 'vehicle_use', 'travel_direction', 'maneuver', 'towed_i', 'fire_i', 'occupant_cnt', 'exceed_speed_limit_i', 'towed_by', 'towed_to', 'area_00_i', 'area_01_i', 'area_02_i', 'area_03_i', 'area_04_i', 'area_05_i', 'area_06_i', 'area_07_i', 'area_08_i', 'area_09_i', 'area_10_i', 'area_11_i', 'area_12_i', 'area_99_i', 'first_contact_point', 'cmv_id', 'usdot_no', 'ccmc_no', 'ilcc_no', 'commercial_src', 'gvwr', 'carrier_name', 'carrier_state', 'carrier_city', 'hazmat_placards_i', 'hazmat_name', 'un_no', 'hazmat_present_i', 'hazmat_report_i', 'hazmat_report_no', 'mcs_report_i', 'mcs_report_no', 'hazmat_vio_cause_crash_i', 'mcs_vio_cause_crash_i', 'idot_permit_no', 'wide_load_i', 'trailer1_width', 'trailer2_width', 'trailer1_length', 'trailer2_length', 'total_vehicle_length', 'axle_cnt', 'veh

In [94]:
print(crashes.columns.tolist())

['crash_record_id', 'crash_date_est_i', 'crash_date', 'posted_speed_limit', 'traffic_control_device', 'device_condition', 'weather_condition', 'lighting_condition', 'first_crash_type', 'trafficway_type', 'lane_cnt', 'alignment', 'roadway_surface_cond', 'road_defect', 'report_type', 'crash_type', 'intersection_related_i', 'private_property_i', 'hit_and_run_i', 'damage', 'date_police_notified', 'prim_contributory_cause', 'sec_contributory_cause', 'street_no', 'street_direction', 'street_name', 'beat_of_occurrence', 'photos_taken_i', 'statements_taken_i', 'dooring_i', 'work_zone_i', 'work_zone_type', 'workers_present_i', 'num_units', 'most_severe_injury', 'injuries_total', 'injuries_fatal', 'injuries_incapacitating', 'injuries_non_incapacitating', 'injuries_reported_not_evident', 'injuries_no_indication', 'injuries_unknown', 'crash_hour', 'crash_day_of_week', 'crash_month', 'latitude', 'longitude', 'location']


In [115]:
vehicles_keep = [
    'crash_record_id',
    'unit_type',
    'vehicle_type',
    'maneuver',
    'exceed_speed_limit_i'
]

vehicles_clean = vehicles[vehicles_keep].copy()

In [116]:
crashes_keep = [
    'crash_record_id',
    'first_crash_type',
    'crash_type',
    'prim_contributory_cause',
    'sec_contributory_cause',
    'posted_speed_limit',
    'weather_condition',
    'lighting_condition',
    'roadway_surface_cond',
    'traffic_control_device',
    'trafficway_type',
    'num_units',
    
]
crashes_clean = crashes[crashes_keep].copy()

#Got rid of variables that may not be needed for analysis, lmk if we should get rid of more. 

In [117]:
crashes_clean.head()

,crash_record_id,first_crash_type,crash_type,prim_contributory_cause,sec_contributory_cause,posted_speed_limit,weather_condition,lighting_condition,roadway_surface_cond,traffic_control_device,trafficway_type,num_units
0,8a82d14f6d2d392638a8c5f5bdaee89ce8c42005c4529c...,TURNING,INJURY AND / OR TOW DUE TO CRASH,FAILING TO YIELD RIGHT-OF-WAY,NOT APPLICABLE,30,CLEAR,"DARKNESS, LIGHTED ROAD",DRY,TRAFFIC SIGNAL,CENTER TURN LANE,2
1,cc89da2a2705cf16fbe17bafd4205fad11bc3853956f41...,FIXED OBJECT,NO INJURY / DRIVE AWAY,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,30,RAIN,"DARKNESS, LIGHTED ROAD",WET,TRAFFIC SIGNAL,ONE-WAY,1
2,3b32c74ced97162dcb27f4cab0e9b3beb389e6548608b7...,SIDESWIPE SAME DIRECTION,NO INJURY / DRIVE AWAY,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,UNABLE TO DETERMINE,30,UNKNOWN,"DARKNESS, LIGHTED ROAD",UNKNOWN,UNKNOWN,UNKNOWN,2
3,783c000fe66fb73635b07ba9490bd2de72dc1339a96ad6...,ANGLE,INJURY AND / OR TOW DUE TO CRASH,FAILING TO YIELD RIGHT-OF-WAY,UNABLE TO DETERMINE,30,CLEAR,"DARKNESS, LIGHTED ROAD",DRY,STOP SIGN/FLASHER,FOUR WAY,2
4,07aea53a5f52a70521c0738eedb5821a1746b7b996a6ff...,PARKED MOTOR VEHICLE,NO INJURY / DRIVE AWAY,UNABLE TO DETERMINE,NOT APPLICABLE,30,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,2


In [118]:
print(crashes_clean.shape)

print(vehicles_clean.shape)

(112053, 12)
(228310, 5)


In [119]:
crashes.head()

,crash_record_id,crash_date_est_i,crash_date,posted_speed_limit,traffic_control_device,device_condition,weather_condition,lighting_condition,first_crash_type,trafficway_type,...,injuries_non_incapacitating,injuries_reported_not_evident,injuries_no_indication,injuries_unknown,crash_hour,crash_day_of_week,crash_month,latitude,longitude,location
0,8a82d14f6d2d392638a8c5f5bdaee89ce8c42005c4529c...,NaN,2024-12-31T23:47:00.000,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,"DARKNESS, LIGHTED ROAD",TURNING,CENTER TURN LANE,...,0.0,0.0,3.0,0.0,23,3,12,41.722380,-87.575207,POINT (-87.57520690929 41.722379604639)
1,cc89da2a2705cf16fbe17bafd4205fad11bc3853956f41...,NaN,2024-12-31T23:41:00.000,30,TRAFFIC SIGNAL,UNKNOWN,RAIN,"DARKNESS, LIGHTED ROAD",FIXED OBJECT,ONE-WAY,...,0.0,0.0,1.0,0.0,23,3,12,41.891133,-87.619160,POINT (-87.619160207046 41.891133170855)
2,3b32c74ced97162dcb27f4cab0e9b3beb389e6548608b7...,NaN,2024-12-31T23:30:00.000,30,UNKNOWN,UNKNOWN,UNKNOWN,"DARKNESS, LIGHTED ROAD",SIDESWIPE SAME DIRECTION,UNKNOWN,...,0.0,0.0,3.0,0.0,23,3,12,41.886703,-87.624114,POINT (-87.624113508031 41.886703274211)
3,783c000fe66fb73635b07ba9490bd2de72dc1339a96ad6...,NaN,2024-12-31T23:18:00.000,30,STOP SIGN/FLASHER,FUNCTIONING PROPERLY,CLEAR,"DARKNESS, LIGHTED ROAD",ANGLE,FOUR WAY,...,0.0,0.0,2.0,0.0,23,3,12,41.736786,-87.601330,POINT (-87.60132951938 41.736786025858)
4,07aea53a5f52a70521c0738eedb5821a1746b7b996a6ff...,NaN,2024-12-31T23:00:00.000,30,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,PARKED MOTOR VEHICLE,UNKNOWN,...,0.0,0.0,1.0,0.0,23,3,12,42.014422,-87.812556,POINT (-87.812556237548 42.014421503701)


In [120]:
merged = pd.merge(
    crashes_clean,
    vehicles_clean,
    on="crash_record_id",
    how="inner"
)
#The datasets were linked using CRASH_RECORD_ID, which represents a one-to-many relationship where a single crash may involve multiple vehicles.
#Due to the one-to-many relationship between crashes and vehicles, the primary dataset is structured at the vehicle level to capture maneuver-specific behavior.

In [121]:
merged.to_csv("../data/processed/merged_2024.csv", index=False)
#making merged file in processed data folder. 

In [122]:
merged

,crash_record_id,first_crash_type,crash_type,prim_contributory_cause,sec_contributory_cause,posted_speed_limit,weather_condition,lighting_condition,roadway_surface_cond,traffic_control_device,trafficway_type,num_units,unit_type,vehicle_type,maneuver,exceed_speed_limit_i
0,8a82d14f6d2d392638a8c5f5bdaee89ce8c42005c4529c...,TURNING,INJURY AND / OR TOW DUE TO CRASH,FAILING TO YIELD RIGHT-OF-WAY,NOT APPLICABLE,30,CLEAR,"DARKNESS, LIGHTED ROAD",DRY,TRAFFIC SIGNAL,CENTER TURN LANE,2,DRIVER,PASSENGER,TURNING LEFT,NaN
1,8a82d14f6d2d392638a8c5f5bdaee89ce8c42005c4529c...,TURNING,INJURY AND / OR TOW DUE TO CRASH,FAILING TO YIELD RIGHT-OF-WAY,NOT APPLICABLE,30,CLEAR,"DARKNESS, LIGHTED ROAD",DRY,TRAFFIC SIGNAL,CENTER TURN LANE,2,DRIVER,PASSENGER,STRAIGHT AHEAD,NaN
2,cc89da2a2705cf16fbe17bafd4205fad11bc3853956f41...,FIXED OBJECT,NO INJURY / DRIVE AWAY,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,30,RAIN,"DARKNESS, LIGHTED ROAD",WET,TRAFFIC SIGNAL,ONE-WAY,1,DRIVER,PASSENGER,STRAIGHT AHEAD,NaN
3,3b32c74ced97162dcb27f4cab0e9b3beb389e6548608b7...,SIDESWIPE SAME DIRECTION,NO INJURY / DRIVE AWAY,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,UNABLE TO DETERMINE,30,UNKNOWN,"DARKNESS, LIGHTED ROAD",UNKNOWN,UNKNOWN,UNKNOWN,2,DRIVER,PASSENGER,MERGING,NaN
4,3b32c74ced97162dcb27f4cab0e9b3beb389e6548608b7...,SIDESWIPE SAME DIRECTION,NO INJURY / DRIVE AWAY,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,UNABLE TO DETERMINE,30,UNKNOWN,"DARKNESS, LIGHTED ROAD",UNKNOWN,UNKNOWN,UNKNOWN,2,DRIVER,PASSENGER,STRAIGHT AHEAD,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
228305,3b5f1493c58026c2313460771fd30a5019b95a909e11fb...,PARKED MOTOR VEHICLE,NO INJURY / DRIVE AWAY,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,NOT APPLICABLE,30,CLEAR,"DARKNESS, LIGHTED ROAD",WET,NO CONTROLS,NOT DIVIDED,2,PARKED,PASSENGER,PARKED,NaN
228306,9be334609035449937964d28dbc3a6ee198a18b64bdea5...,PARKED MOTOR VEHICLE,NO INJURY / DRIVE AWAY,FOLLOWING TOO CLOSELY,NOT APPLICABLE,30,CLEAR,DARKNESS,UNKNOWN,NO CONTROLS,NOT DIVIDED,2,DRIVER,UNKNOWN/NA,UNKNOWN/NA,NaN
228307,9be334609035449937964d28dbc3a6ee198a18b64bdea5...,PARKED MOTOR VEHICLE,NO INJURY / DRIVE AWAY,FOLLOWING TOO CLOSELY,NOT APPLICABLE,30,CLEAR,DARKNESS,UNKNOWN,NO CONTROLS,NOT DIVIDED,2,PARKED,VAN/MINI-VAN,PARKED,NaN
228308,ce46f735c5c4d216ef355f25b4159dbc97637fbb3431b0...,PARKED MOTOR VEHICLE,NO INJURY / DRIVE AWAY,UNABLE TO DETERMINE,UNABLE TO DETERMINE,30,UNKNOWN,UNKNOWN,UNKNOWN,NO CONTROLS,OTHER,2,DRIVER,UNKNOWN/NA,UNKNOWN/NA,NaN


In [123]:
#Do white space normalization, case inconcistancies, format inconc, typographical erors, data type mismatch
#tiered cleaning method, Deep cleaning for vehicle and crash variables and mod for contextual variables


In [124]:
# strip and standardize all string columns
for col in merged.select_dtypes(include="object").columns:
    merged[col] = merged[col].astype(str).str.strip().str.upper()

#Get rid of N/A / missing in Manuever, most important variable.
merged["maneuver"] = merged["maneuver"].replace("NAN", pd.NA)
merged["maneuver"] = merged["maneuver"].fillna("UNKNOWN")
merged["maneuver"] = merged["maneuver"].replace("UNKNOWN/NA", "UNKNOWN")

In [125]:
merged

,crash_record_id,first_crash_type,crash_type,prim_contributory_cause,sec_contributory_cause,posted_speed_limit,weather_condition,lighting_condition,roadway_surface_cond,traffic_control_device,trafficway_type,num_units,unit_type,vehicle_type,maneuver,exceed_speed_limit_i
0,8A82D14F6D2D392638A8C5F5BDAEE89CE8C42005C4529C...,TURNING,INJURY AND / OR TOW DUE TO CRASH,FAILING TO YIELD RIGHT-OF-WAY,NOT APPLICABLE,30,CLEAR,"DARKNESS, LIGHTED ROAD",DRY,TRAFFIC SIGNAL,CENTER TURN LANE,2,DRIVER,PASSENGER,TURNING LEFT,NAN
1,8A82D14F6D2D392638A8C5F5BDAEE89CE8C42005C4529C...,TURNING,INJURY AND / OR TOW DUE TO CRASH,FAILING TO YIELD RIGHT-OF-WAY,NOT APPLICABLE,30,CLEAR,"DARKNESS, LIGHTED ROAD",DRY,TRAFFIC SIGNAL,CENTER TURN LANE,2,DRIVER,PASSENGER,STRAIGHT AHEAD,NAN
2,CC89DA2A2705CF16FBE17BAFD4205FAD11BC3853956F41...,FIXED OBJECT,NO INJURY / DRIVE AWAY,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,30,RAIN,"DARKNESS, LIGHTED ROAD",WET,TRAFFIC SIGNAL,ONE-WAY,1,DRIVER,PASSENGER,STRAIGHT AHEAD,NAN
3,3B32C74CED97162DCB27F4CAB0E9B3BEB389E6548608B7...,SIDESWIPE SAME DIRECTION,NO INJURY / DRIVE AWAY,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,UNABLE TO DETERMINE,30,UNKNOWN,"DARKNESS, LIGHTED ROAD",UNKNOWN,UNKNOWN,UNKNOWN,2,DRIVER,PASSENGER,MERGING,NAN
4,3B32C74CED97162DCB27F4CAB0E9B3BEB389E6548608B7...,SIDESWIPE SAME DIRECTION,NO INJURY / DRIVE AWAY,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,UNABLE TO DETERMINE,30,UNKNOWN,"DARKNESS, LIGHTED ROAD",UNKNOWN,UNKNOWN,UNKNOWN,2,DRIVER,PASSENGER,STRAIGHT AHEAD,NAN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
228305,3B5F1493C58026C2313460771FD30A5019B95A909E11FB...,PARKED MOTOR VEHICLE,NO INJURY / DRIVE AWAY,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,NOT APPLICABLE,30,CLEAR,"DARKNESS, LIGHTED ROAD",WET,NO CONTROLS,NOT DIVIDED,2,PARKED,PASSENGER,PARKED,NAN
228306,9BE334609035449937964D28DBC3A6EE198A18B64BDEA5...,PARKED MOTOR VEHICLE,NO INJURY / DRIVE AWAY,FOLLOWING TOO CLOSELY,NOT APPLICABLE,30,CLEAR,DARKNESS,UNKNOWN,NO CONTROLS,NOT DIVIDED,2,DRIVER,UNKNOWN/NA,UNKNOWN,NAN
228307,9BE334609035449937964D28DBC3A6EE198A18B64BDEA5...,PARKED MOTOR VEHICLE,NO INJURY / DRIVE AWAY,FOLLOWING TOO CLOSELY,NOT APPLICABLE,30,CLEAR,DARKNESS,UNKNOWN,NO CONTROLS,NOT DIVIDED,2,PARKED,VAN/MINI-VAN,PARKED,NAN
228308,CE46F735C5C4D216EF355F25B4159DBC97637FBB3431B0...,PARKED MOTOR VEHICLE,NO INJURY / DRIVE AWAY,UNABLE TO DETERMINE,UNABLE TO DETERMINE,30,UNKNOWN,UNKNOWN,UNKNOWN,NO CONTROLS,OTHER,2,DRIVER,UNKNOWN/NA,UNKNOWN,NAN


In [126]:
merged["maneuver"].value_counts(dropna=False).head(20)

maneuver
STRAIGHT AHEAD                        108406
PARKED                                 29732
UNKNOWN                                21075
TURNING LEFT                           14133
SLOW/STOP IN TRAFFIC                   13388
TURNING RIGHT                           8161
BACKING                                 8158
PASSING/OVERTAKING                      5700
CHANGING LANES                          4092
OTHER                                   3981
ENTERING TRAFFIC LANE FROM PARKING      2715
MERGING                                 1831
U-TURN                                  1402
LEAVING TRAFFIC LANE TO PARK            1120
STARTING IN TRAFFIC                     1031
AVOIDING VEHICLES/OBJECTS                861
PARKED IN TRAFFIC LANE                   692
SKIDDING/CONTROL LOSS                    536
ENTER FROM DRIVE/ALLEY                   501
DRIVING WRONG WAY                        392
Name: count, dtype: int64

In [127]:
merged["prim_contributory_cause"].value_counts().head(20)


prim_contributory_cause
UNABLE TO DETERMINE                                                                 93111
FAILING TO YIELD RIGHT-OF-WAY                                                       26901
FOLLOWING TOO CLOSELY                                                               20351
IMPROPER OVERTAKING/PASSING                                                         12568
NOT APPLICABLE                                                                      11311
FAILING TO REDUCE SPEED TO AVOID CRASH                                               9653
DRIVING SKILLS/KNOWLEDGE/EXPERIENCE                                                  8636
IMPROPER TURNING/NO SIGNAL                                                           8084
IMPROPER LANE USAGE                                                                  7626
IMPROPER BACKING                                                                     7034
DISREGARDING TRAFFIC SIGNALS                                                

In [128]:
#reducing unable to determine and N/A to unknown, easier for analysis. 
merged["prim_contributory_cause"] = merged["prim_contributory_cause"].replace({
    "UNABLE TO DETERMINE": "UNKNOWN",
    "NOT APPLICABLE": "UNKNOWN"
}) 
merged["sec_contributory_cause"] = merged["sec_contributory_cause"].replace({
    "UNABLE TO DETERMINE": "UNKNOWN",
    "NOT APPLICABLE": "UNKNOWN"
}) 

#Grouping some similiar causes helps aid with better and cleaner analysis
def clean_cause(x):
    if pd.isna(x):
        return "UNKNOWN"
    
    if "DISTRACTION" in x or "TEXTING" in x or "CELL PHONE" in x:
        return "DISTRACTION"
    
    if "DISREGARDING" in x:
        return "TRAFFIC_VIOLATION"
    
    if "DRINK" in x or "ALCOHOL" in x or "DRUG" in x:
        return "IMPAIRMENT"
    
    return x
merged["prim_cause_cl"] = merged["prim_contributory_cause"].apply(clean_cause)
merged["secnd_cause_cl"] = merged["sec_contributory_cause"].apply(clean_cause)

In [129]:
merged.columns 


Index(['crash_record_id', 'first_crash_type', 'crash_type',
       'prim_contributory_cause', 'sec_contributory_cause',
       'posted_speed_limit', 'weather_condition', 'lighting_condition',
       'roadway_surface_cond', 'traffic_control_device', 'trafficway_type',
       'num_units', 'unit_type', 'vehicle_type', 'maneuver',
       'exceed_speed_limit_i', 'prim_cause_cl', 'secnd_cause_cl'],
      dtype='object')

In [136]:
#cleaning for other columns, they will all get basic cleaning like as before, in case items were not cleaned correctly.

#crash types
for col in ["first_crash_type", "crash_type"]:
    merged[col] = merged[col].astype(str).str.strip().str.upper()

#speed limit - issues with this maybe fix later. Add to issues part of report. 
# many posted limits are not real/ prolyl placeholders, what should we do with this, or remove column, or leave as is. note in analysis. 
merged["posted_speed_limit"] = pd.to_numeric(
    merged["posted_speed_limit"], errors="coerce"
)

#Environmental conditions
env_cols = [
    "weather_condition",
    "lighting_condition",
    "roadway_surface_cond"
]

for col in env_cols:
    merged[col] = merged[col].astype(str).str.strip().str.upper()

#Roadways
road_cols = ["traffic_control_device", "trafficway_type"]

for col in road_cols:
    merged[col] = merged[col].astype(str).str.strip().str.upper()
#Num units
merged["num_units"] = pd.to_numeric(merged["num_units"], errors="coerce")

#vehicle info
veh_cols = ["unit_type", "vehicle_type"]

for col in veh_cols:
    merged[col] = merged[col].astype(str).str.strip().str.upper()


#speed limit - issues with this maybe fix later. Add to issues part of report. 
# many posted limits are not real/ prolyl placeholders, what should we do with this, or remove column, or leave as is. note in analysis. 
#another issue to include in phase 1. The exceed speed limit indicator has 200000+ N/A values. Should we drop both columns, as they wont be useful. 
#INCLUDE BOTH FOR ISSUES RAN INTO/ discuss how we plan to address later on. 
merged["posted_speed_limit"] = pd.to_numeric(
    merged["posted_speed_limit"], errors="coerce"
)
#replace clean cause with original
merged = merged.drop(columns=[
    "prim_contributory_cause",
    "sec_contributory_cause"
])

merged = merged.rename(columns={
    "prim_cause_cl": "primary_cause",
    "secnd_cause_cl": "secondary_cause"
})

#creating a seconadry analysis dataframe that only includes instances where manuever is not UNKNOWN.
#Another good issue to include is that exaclty, what to do when manuever is unknown, We solved by making 2 seperate tables. 
analysis_df = merged[
    (merged["maneuver"] != "UNKNOWN")
]

In [137]:
analysis_df

,crash_record_id,first_crash_type,crash_type,posted_speed_limit,weather_condition,lighting_condition,roadway_surface_cond,traffic_control_device,trafficway_type,num_units,unit_type,vehicle_type,maneuver,exceed_speed_limit_i,primary_cause,secondary_cause
0,8A82D14F6D2D392638A8C5F5BDAEE89CE8C42005C4529C...,TURNING,INJURY AND / OR TOW DUE TO CRASH,30,CLEAR,"DARKNESS, LIGHTED ROAD",DRY,TRAFFIC SIGNAL,CENTER TURN LANE,2,DRIVER,PASSENGER,TURNING LEFT,NAN,FAILING TO YIELD RIGHT-OF-WAY,UNKNOWN
1,8A82D14F6D2D392638A8C5F5BDAEE89CE8C42005C4529C...,TURNING,INJURY AND / OR TOW DUE TO CRASH,30,CLEAR,"DARKNESS, LIGHTED ROAD",DRY,TRAFFIC SIGNAL,CENTER TURN LANE,2,DRIVER,PASSENGER,STRAIGHT AHEAD,NAN,FAILING TO YIELD RIGHT-OF-WAY,UNKNOWN
2,CC89DA2A2705CF16FBE17BAFD4205FAD11BC3853956F41...,FIXED OBJECT,NO INJURY / DRIVE AWAY,30,RAIN,"DARKNESS, LIGHTED ROAD",WET,TRAFFIC SIGNAL,ONE-WAY,1,DRIVER,PASSENGER,STRAIGHT AHEAD,NAN,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE
3,3B32C74CED97162DCB27F4CAB0E9B3BEB389E6548608B7...,SIDESWIPE SAME DIRECTION,NO INJURY / DRIVE AWAY,30,UNKNOWN,"DARKNESS, LIGHTED ROAD",UNKNOWN,UNKNOWN,UNKNOWN,2,DRIVER,PASSENGER,MERGING,NAN,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,UNKNOWN
4,3B32C74CED97162DCB27F4CAB0E9B3BEB389E6548608B7...,SIDESWIPE SAME DIRECTION,NO INJURY / DRIVE AWAY,30,UNKNOWN,"DARKNESS, LIGHTED ROAD",UNKNOWN,UNKNOWN,UNKNOWN,2,DRIVER,PASSENGER,STRAIGHT AHEAD,NAN,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,UNKNOWN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
228303,89428833103E1E56232FAB1A8D97EE7CEE92C72C9C80A0...,SIDESWIPE SAME DIRECTION,NO INJURY / DRIVE AWAY,30,CLEAR,DARKNESS,DRY,TRAFFIC SIGNAL,NOT DIVIDED,2,DRIVER,PASSENGER,STRAIGHT AHEAD,NAN,UNKNOWN,UNKNOWN
228304,3B5F1493C58026C2313460771FD30A5019B95A909E11FB...,PARKED MOTOR VEHICLE,NO INJURY / DRIVE AWAY,30,CLEAR,"DARKNESS, LIGHTED ROAD",WET,NO CONTROLS,NOT DIVIDED,2,DRIVER,UNKNOWN/NA,BACKING,NAN,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,UNKNOWN
228305,3B5F1493C58026C2313460771FD30A5019B95A909E11FB...,PARKED MOTOR VEHICLE,NO INJURY / DRIVE AWAY,30,CLEAR,"DARKNESS, LIGHTED ROAD",WET,NO CONTROLS,NOT DIVIDED,2,PARKED,PASSENGER,PARKED,NAN,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,UNKNOWN
228307,9BE334609035449937964D28DBC3A6EE198A18B64BDEA5...,PARKED MOTOR VEHICLE,NO INJURY / DRIVE AWAY,30,CLEAR,DARKNESS,UNKNOWN,NO CONTROLS,NOT DIVIDED,2,PARKED,VAN/MINI-VAN,PARKED,NAN,FOLLOWING TOO CLOSELY,UNKNOWN
